# Germline Whole Exome Sequencing (WES): Variant Discovery

**Author:** Eman Koosehlar

**Notebook:** 1 of the series  

**Version:** *1.0*

**Last Updated:** *June 2026*

**License:** MIT License


---

### Runtime Requirements

This notebook is designed to run on the default Google Colab runtime.

**Recommended runtime**
- Runtime type: `CPU`

No GPU or TPU is required.

---

## Learning Journey

This notebook is the first part of the <font color = "redish">**Germline Whole Exome Sequencing (WES) Analysis Series**</font>.

The goal of this notebook is to demonstrate how raw sequencing reads (FASTQ files) are processed through quality control, alignment, post-processing, variant calling, and hard filtering to produce a high-confidence VCF file suitable for downstream analysis.

This notebook focuses on **variant discovery** rather than biological interpretation. The resulting filtered VCF will serve as the starting point for variant annotation, prioritization, and clinical interpretation in the subsequent notebooks of this series.


## Overview
This notebook demonstrates a complete germline Whole Exome Sequencing (WES) variant calling workflow using GATK Best Practices.

The pipeline starts from raw FASTQ files and ends with a filtered VCF file containing high-confidence variants.

## Pipeline Overview

<img src="../figures/WES_Pipeline.png" width="800">


> <span style="color:red"> This notebook is intended for educational and research purposes.</span>

**Table of contents**<a id='toc0_'></a>    
- [Introduction](#toc1_)
- [Requirements (Tools/Environment/databases)](#toc2_)  
  - [Installation](#toc2_1_)
  - [Environment Settings](#toc2_2_)
  - [Download required files](#toc2_3_)
- [Step 1: FastQC](#toc3_)
- [Step 2: Trimmomatic (Optional)](#toc4_)
- [Step 3: Alignment with BWA-MEM](#toc5_)
- [Step 4: Sort BAM](#toc6_)
- [Step 5: MarkDuplicates](#toc7_)
- [Step 6: Base Quality Score Recalibration (BQSR)](#toc8_)
  - [Apply BQSR](#toc8_1_)
- [Step 7: Variant Calling using HaplotypeCaller](#toc9_)
- [Step 8: Hard Filtering](#toc10_)
- [Summary](#toc11_)
- [References](#toc12_)


<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=2
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

## <a id='toc1_'></a>[Introduction](#toc0_)

Whole Exome Sequencing (WES) focuses on sequencing the **protein-coding regions** of the genome (the exome), which represent approximately `1–2%` of the human genome but contain the majority of known disease-causing variants.

The purpose of this workflow is to identify small genetic variants (SNVs and Indels) from exome sequencing data.

This notebook follows the general recommendations of the `GATK Best Practices` pipeline for germline variant discovery.

---


## <a id='toc2_'></a>[Requirements (Environment/Tools/databases)](#toc0_)



The following tools are required:

| Tool | Purpose |
|--------|---------|
| FastQC | Read quality assessment |
| Multiqc | Create Interactive report |
| Trimmomatic | Adapter and quality trimming |
| BWA-MEM | Alignment |
| Samtools | BAM processing |
| GATK | Variant calling |
| Picard | Duplicate marking |

 **Project Directories**

To keep the workflow organized and reproducible, all files generated during the analysis are stored in dedicated project directories.

```text
WES_Project/
│
│
├── Reference/ (✅Current Notebook)
│   ├── Reference_Genome hg38
│   └── Gatk_Known_Sites
│       ├── dbsnp138
│       ├── hapmap_3.3
│       ├── 1000G_omni2.5.hg38
│       ├── 1000G_phase1.snps.high_confidence.hg38
│       ├── Homo_sapiens_assembly38.known_indels
│       ├── Mills_and_1000G_gold_standard.indels.hg38
│       ├── wgs_calling_regions.hg38.interval_list
│       └── hg38.scattered_calling_intervals/  
│ 
│ 
│ 
├── Alignment/ (✅Current Notebook)
│   └── Mapped Reads Files
│
│
├── Germline_Variant_Calls/ (✅Current Notebook)
│   └── Raw VCF files
│
│
├── VCF/ (✅Current Notebook)
│   ├── Filtered VCF files
│   └── Final VCF used for annotation
│
│
├── Annotation/ (🚧 In 02_Germline_Variant_Annotation Notebook)
│   ├── Annotated VCF files
│   └── Annotation reports
│
│
├── Prioritization/ (🚧 In 02_Germline_Variant_Annotation Notebook)
│   └── Filtered annotated VCF files ready for downstream interpretation
│
│
└── Results/ (🚧 In 03_Germline_Variant_Interpretation Notebook)
    └── Summary and final outputs
```

### Directory Description

* **Reference/** – Stores the reference genome and known variant databases required by GATK and VEP.

* **Germline_Variant_Calls/** - Stores VCF files generated after variant calling

* **VCF/** – Stores VCF files generated after hard filtering.

* **Alingment/** – Stores maped reads file after alingning reads to the reference genome



### <a id='toc2_1_'></a>[Installation](#toc0_)

In [ ]:
# Java
!apt install openjdk-17-jdk -y

# FastQC
!apt install fastqc -y

# Multiqc
!apt-get install multiqc

# Samtools
!apt install samtools -y

# Picard
!apt-get install picard

# BWA
!apt install bwa -y

# Trimmomatic
!apt install trimmomatic -y

# Download GATK
!wget -q https://github.com/broadinstitute/gatk/releases/download/4.6.2.0/gatk-4.6.2.0.zip

!unzip -q gatk-4.6.2.0.zip

- Check the tools with the fullowing commands.

In [ ]:
!fastqc --help

In [ ]:
!samtools --help

In [ ]:
!bwa

In [ ]:
!gatk --help

In [ ]:
!java -jar /content/picard.jar -h

----
- **Optional installation for samtools (if the above section had error use this)**


!wget https://github.com/samtools/samtools/releases/download/1.7/samtools-1.7.tar.bz2


!tar --bzip2 -xvf samtools-1.7.tar.bz2

cd samtools-1.7/

!make

!make install

cd /content

!rm samtools-1.7.tar.bz2

`use each command in a separate cells`

- **Picard installation (optional)**


!wget https://github.com/broadinstitute/picard/releases/download/2.18.14/picard.jar




----

### <a id='toc2_2_'></a>[Environment settings](#toc0_)

 Environment setting and creating required directories for the follwing steps

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
# Create directory for alingment section
!mkdir WES_Analysis

!mkdir WES_Analysis/Alignment

# Creating dir for genome file
!mkdir -p WES_Analysis/Reference/Reference_Genome

# Creating dir for Known site Databases
!mkdir WES_Analysis/Reference/Gatk_Known_Sites

# Create dir for VCF files
!mkdir WES_Analysis/Germline_Variant_Calls WES_Analysis/VCF



In [ ]:
#Environment settings

CHRS = 'chr6_and_chr17'
GATK_REGIONS='-L chr6 -L chr17'
Known_Sites = '/content/WES_Analysis/Reference/Gatk_Known_Sites'
Germline_Calls = '/content/WES_Analysis/Germline_Variant_Calls'
VCF = '/content/WES_Analysis/VCF'
Reference_Genome = '/content/WES_Analysis/Reference/Reference_Genome'
Alignment = '/content/WES_Analysis/Alignment'


### <a id='toc2_3_'></a>[Download required files](#toc0_)

1. **Raw Fastq files**

In [ ]:
# Download and extract fastq files
!wget https://genomedata.org/pmbio-workshop/fastqs/chr6_and_chr17/Exome_Norm.tar
!tar -xvf Exome_Norm.tar


- *Some exploration on raw data (optional)*


In [ ]:
# look at the header of the file (same proccess for the R2 file)
!zcat Exome_Norm/Exome_Norm_R1.fastq.gz | head

# what do R1 and R2 refer to? What is the length of each read?
!zcat Exome_Norm/Exome_Norm_R1.fastq.gz | head -n 2 | tail -n 1 | wc

# how many lines are there in the Exome_Tumor file
!zcat Exome_Norm/Exome_Norm_R1.fastq.gz | wc -l # There are: 33,326,620

# how many paired reads or fragments are there then?
!expr 25466068 / 4 # There are: 8,331,655 paired end reads

# how many total bases of data are in the Exome Tumor data set?
!echo "6366517 * (101 * 2)" | bc # There are: 1,682,994,310 bases of data

# how many total bases when expressed as "gigabases" (specify 2 decimal points using `scale`)
!echo "scale=2; (6366517 * (101 * 2))/1000000000" | bc # There are: 1.68 Gbp of data


2. **Downloding Reference Genome**

In [ ]:
# Downloading refrence genome
!wget -P {Reference_Genome}  http://genomedata.org/pmbio-workshop/references/genome/$CHRS/ref_genome.tar

In [ ]:
# Extracting ref genome
!tar -xvf {Reference_Genome}/ref_genome.tar

In [ ]:
# Removing unused tar file
rm {Reference_Genome}/ref_genome.tar

In [ ]:
# Changing directory to /content
cd /content/

In [ ]:
# move all extracted ref files to its directories
!mv ref_genome.* {Reference_Genome}

In [ ]:
# Unzip ref genome file
!gunzip {Reference_Genome}/ref_genome.fa.gz

- *Checking ref genome file contents:*

In [ ]:
# Inspecting header of the ref genome file
!head {Reference_Genome}/ref_genome.fa

In [ ]:
# How long are to two chromosomes combined (in bases and Mbp)? Use grep to skip the header lines for each chromosome.
!grep -v ">" {Reference_Genome}/ref_genome.fa | wc


# How long does that command take to run?
!time grep -v ">" {Reference_Genome}/ref_genome.fa | wc

# View 10 lines from approximately the middle of this file
!head -n 2500000 {Reference_Genome}/ref_genome.fa | tail


In [ ]:
# What is the count of each base in the entire reference genome file (skipping the header lines for each sequence)?


from collections import Counter

base_counts = Counter()

with open('/content/WES_Analysis/Reference/Reference_Genome/ref_genome.fa') as f:
    for line in f:
        if line.startswith('>'):
            continue
        base_counts.update(line.strip())

# Print sorted base counts
for base in sorted(base_counts):
    print(f"{base} {base_counts[base]}")




In [ ]:
# How many time a pattern like 'GAATTC' occurs this genome?

import re

pattern = re.compile(r'(?=GAATTC)')  # Lookahead to allow overlapping matches
chunks = []

# Read and collect all lines except headers
with open('/content/WES_Analysis/Reference/Reference_Genome/ref_genome.fa') as f:
    for line in f:
        if not line.startswith('>'):
            chunks.append(line.strip())

sequence = ''.join(chunks)  # Efficient one-time join
count = len(pattern.findall(sequence))

print(f"The pattern 'GAATTC' occurs {count} times in the genome.")

# The pattern 'GAATTC' occurs 71525 times in the genome.

3. **Downloading Known sites databases**

In [ ]:
cd {Known_Sites}

In [ ]:
!gsutil -o "GSUtil:check_hashes=never" cp gs://genomics-public-data/resources/broad/hg38/v0/Homo_sapiens_assembly38.dbsnp138.vcf .

In [ ]:
!gzip {Known_Sites}/Homo_sapiens_assembly38.dbsnp138.vcf

In [ ]:
!gsutil cp gs://genomics-public-data/resources/broad/hg38/v0/hapmap_3.3.hg38.vcf.gz .

In [ ]:
!gsutil cp gs://genomics-public-data/resources/broad/hg38/v0/1000G_omni2.5.hg38.vcf.gz .
!gsutil cp gs://genomics-public-data/resources/broad/hg38/v0/1000G_phase1.snps.high_confidence.hg38.vcf.gz .

In [ ]:
# Indel calibration call sets - dbsnp, Mills
!gsutil cp gs://genomics-public-data/resources/broad/hg38/v0/Homo_sapiens_assembly38.known_indels.vcf.gz .
!gsutil cp gs://genomics-public-data/resources/broad/hg38/v0/Mills_and_1000G_gold_standard.indels.hg38.vcf.gz .

In [ ]:
# Interval lists that can be used to parallelize certain GATK tasks
!gsutil cp gs://genomics-public-data/resources/broad/hg38/v0/wgs_calling_regions.hg38.interval_list .
!gsutil cp -r gs://genomics-public-data/resources/broad/hg38/v0/scattered_calling_intervals/ .

In [ ]:
!gunzip {Known_Sites}/Homo_sapiens_assembly38.dbsnp138.vcf.gz

**<span style = 'color:blue'> Outputs generated during this notebook can be stored in:</span>**

- Google Drive (*recommended*)
- Local Colab runtime (*temporary*)

Important intermediate files such as reference genomes,
`BAM` files, and `filtered VCF` files can be reused in
downstream annotation and variant prioritization notebooks.

---

## <a id='toc3_'></a>[Step 1: FastQC](#toc0_)

**Purpose**

FastQC evaluates the quality of raw sequencing reads before alignment.

Important metrics include:

- Per-base sequence quality
- Adapter contamination
- GC content
- Sequence duplication levels

**Why is this important?**

Poor quality reads may lead to:

- Alignment errors
- False-positive variants
- Reduced variant calling accuracy

**Input files:**

- R1 FASTQ
- R2 FASTQ


**Expected output:**

- Quality Control's HTML Reports 

In [ ]:
# Change to the working directory
cd /content/

In [ ]:
# Run fastqc
!fastqc Exome_Norm/Exome_Norm*.fastq.gz -o Exome_Norm/FASTQC/

Check the results by download and view them in your browser. 

- **QC Interpretation**

For WES data, we typically expect:

✓ Per-base quality scores > Q30

✓ Low adapter contamination

✓ Consistent GC distribution

If the reports are acceptable, trimming may not be necessary.

---

## <a id='toc4_'></a>[Step 2: Trimmomatic (Optional)](#toc0_)

**Purpose**

Trimmomatic removes:

- Adapter sequences
- Low-quality bases
- Poor-quality reads

**Is it required?**

Not always.

In this project, FastQC showed high-quality reads with minimal adapter contamination, therefore trimming was not required.

However, this step is included for completeness because many datasets benefit from trimming before alignment.

In [ ]:
!trimmomatic PE \
Exome_Norm/Exome_Norm_R1.fastq.gz Exome_Norm/Exome_Norm_R2.fastq.gz \
R1_paired.fastq.gz R1_unpaired.fastq.gz \
R2_paired.fastq.gz R2_unpaired.fastq.gz \
SLIDINGWINDOW:4:20 MINLEN:50

---